In [ ]:
# Import libraries here
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import pickle
from IPython.display import VimeoVideo
from scipy.stats.mstats import trimmed_var
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [ ]:
df = pd.read_csv("data/SCFP2019.csv.gz")
print("df shape:", df.shape)
df.head()


In [ ]:
prop_biz_owners = df['HBUS'].value_counts(normalize=True)[1]
print("proportion of business owners in df:", prop_biz_owners)


In [ ]:
inccat_dict = {
    1: "0-20",
    2: "21-39.9",
    3: "40-59.9",
    4: "60-79.9",
    5: "80-89.9",
    6: "90-100",
}
df_inccat = (df['INCCAT'].replace(inccat_dict)
            .groupby(df['HBUS'])
            .value_counts(normalize=True)
            .rename("frequency")
            .to_frame()
            .reset_index()
)

df_inccat.head()


In [ ]:
fig, ax = plt.subplots()

sns.barplot(
    data=df_inccat,
    x="INCCAT",
    y="frequency",
    hue="HBUS",
    order=inccat_dict.values(),
    ax=ax
)

ax.set_title("Income Distribution: Business Owners vs. Non-Business Owners")
ax.set_xlabel("Income Category")
ax.set_ylabel("Frequency (%)")

plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x="DEBT",
    y="HOUSES",
    hue="HBUS",
    palette="deep",
    ax=ax
)

ax.set_xlabel("Household Debt")
ax.set_ylabel("Home Value")
ax.set_title("Home Value vs. Household Debt")

plt.show()


In [ ]:
mask = (df['HBUS']==1) & (df['INCOME']<= 5e5) 
df_small_biz = df[mask] # use the column `mask` defined above
print("df_small_biz shape:", df_small_biz.shape)
df_small_biz.head()


In [ ]:
# Plot histogram of "AGE"
fig, ax = plt.subplots()

df_small_biz['AGE'].plot(
    kind='hist',
    bins=10,
    ax=ax
)

ax.set_title('Small Business Owners: Age Distribution')
ax.set_xlabel('Age')
ax.set_ylabel('Frequency (count)')

plt.show()


In [ ]:
# Calculate variance, get 10 largest features
top_ten_var = df_small_biz.var(numeric_only=True).sort_values().tail(10)
top_ten_var


In [ ]:
# Calculate trimmed variance
top_ten_trim_var = (
    df_small_biz
    .apply(trimmed_var,
    limits=(0.1,0.1))
    .sort_values().tail(10)
)

top_ten_trim_var


In [ ]:
# Create horizontal bar chart of `top_ten_trim_var`
fig = px.bar(
    x=top_ten_trim_var,
    y=top_ten_trim_var.index,
    title="Small Business Owners: High Variance Features"
)

fig.update_layout(xaxis_title="Trimmed Variance [$]", yaxis_title="Feature")
# Convert values to billions
#fig.update_traces(x=top_ten_trim_var / 1e9)
fig.show()


In [ ]:
high_var_cols = top_ten_trim_var.tail(5).index.to_list()
high_var_cols


In [ ]:
X = df_small_biz[high_var_cols]
print("X shape:", X.shape)
X.head()


In [ ]:
n_clusters = range(2,13)
inertia_errors = []
silhouette_scores = []

# Add `for` loop to train model and calculate inertia, silhouette score.
for k in n_clusters:
    # build model 
    model = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42)
        )
    
    # train model
    model.fit(X)
    
    # Calculatw inertia
    inertia_errors.append(model.named_steps['kmeans'].inertia_)
    
    # Calculate silhouette_scores
    silhouette_scores.append(
        silhouette_score(X, model.named_steps['kmeans'].labels_)
        )

print("Inertia:", inertia_errors[:11])
print()
print("Silhouette Scores:", silhouette_scores[:3])


In [ ]:
# Create line plot of `inertia_errors` vs `n_clusters`

fig = px.line(
    x= n_clusters, y= inertia_errors, title="K-Means Model: Inertia vs Number of Clusters"
)

fig.update_layout(xaxis_title="Number of Clusters", yaxis_title="Inertia")
fig.show()


In [ ]:
# Create a line plot of `silhouette_scores` vs `n_clusters`

fig = px.line(
    x= n_clusters, y= silhouette_scores, title="K-Means Model: Silhouette Score vs Number of Clusters"
)
fig.update_layout(xaxis_title="Number of Clusters", yaxis_title="Silhouette Score")
fig.show()


In [ ]:
final_model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=3, random_state=42)
)

# Fit model to data
final_model.fit(X)


In [ ]:
labels = final_model.named_steps["kmeans"].labels_
xgb = X.groupby(labels).mean()
xgb


In [ ]:
# Create side-by-side bar chart of `xgb`
fig = px.bar(xgb, barmode="group",
            title="Small Business Owner Finances by Cluster")
fig.update_layout(xaxis_title="Cluster", yaxis_title="Value [$]")

fig.show()


In [ ]:
# Instantiate transformer
pca = PCA(n_components=2,random_state=42)

# Transform `X`
X_t = pca.fit_transform(X)

# Put `X_t` into DataFrame
X_pca = pd.DataFrame(X_t, columns=["PC1","PC2"])

print("X_pca shape:", X_pca.shape)
X_pca.head()


In [ ]:
# Create scatter plot of `PC2` vs `PC1`
fig = px.scatter(
    data_frame=X_pca,
    x="PC1",
    y="PC2",
    color=labels.astype(str),
    title="PCA Representation of Clusters"
)
fig.update_layout(xaxis_title="PC1", yaxis_title="PC2")
fig.show()
